EDA
 ↓
Seasonality adjustment / feature engineering
 ↓
Dense AE
 ↓
LSTM AE
 ↓
LSTM/GRU forecasting anomaly detector
 ↓
Compare anomalies
 ↓
Examine specific dates flagged by each model

#Load historical Dataset

In [ ]:
import pandas as pd
import numpy as np

triangle_daily = pd.read_parquet('data/triangle_pm25_daily_2021_2025.parquet')


| Model                             | What it learns                                 | How anomaly score works             | Worth trying?                  |
| --------------------------------- | ---------------------------------------------- | ----------------------------------- | ------------------------------ |
| **Dense Autoencoder**             | Normal combinations of weather variables       | Reconstruction error                | ✅ Definitely                   |
| **LSTM Autoencoder**              | Normal multi-day weather sequences             | Sequence reconstruction error       | ✅ Definitely                   |
| **Forecasting LSTM/GRU**          | Predicts the next day from prior days          | Prediction residual                 | ✅ Probably my favorite         |
| **Variational Autoencoder (VAE)** | Probabilistic representation of normal weather | Reconstruction + latent probability | ✅ Good advanced model          |
| **1D CNN Autoencoder**            | Short-term local temporal patterns             | Reconstruction error                | 🟡 Interesting comparison      |
| **Transformer Autoencoder**       | Longer-range sequence relationships            | Reconstruction error                | 🟡 Cool, but probably overkill |


Encode sinusoidal seasonality function, PM 2.5 and ozone both have seasonality factors

In [ ]:
air = triangle_daily.copy()

air["date"] = pd.to_datetime(air["date"])
air = air.sort_values("date").reset_index(drop=True)

# Basic calendar variables
air["year"] = air["date"].dt.year
air["month"] = air["date"].dt.month
air["day_of_year"] = air["date"].dt.dayofyear
air["day_of_week"] = air["date"].dt.dayofweek
air["is_weekend"] = (air["day_of_week"] >= 5).astype(int)

# Meteorological season - useful mainly for plots / interpretation
air["season"] = air["month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall"
})

# Cyclical annual seasonality
air["doy_sin"] = np.sin(
    2 * np.pi * air["day_of_year"] / 365.25
)

air["doy_cos"] = np.cos(
    2 * np.pi * air["day_of_year"] / 365.25
)

# Cyclical weekly pattern
air["dow_sin"] = np.sin(
    2 * np.pi * air["day_of_week"] / 7
)

air["dow_cos"] = np.cos(
    2 * np.pi * air["day_of_week"] / 7
)

The data is sequential, not independent or identically distributed. So adding lag to account for previous day's PM 2.5 values

In [ ]:
pm25_lags = [1, 2, 3, 7, 14]

for lag in pm25_lags:
    air[f"pm25_lag_{lag}"] = air["pm25"].shift(lag)

In [ ]:
for window in [3, 7, 14, 30]:

    # shift first so today's PM2.5 is NOT included
    historical_pm25 = air["pm25"].shift(1)

    air[f"pm25_mean_{window}d"] = (
        historical_pm25
        .rolling(window)
        .mean()
    )

    air[f"pm25_std_{window}d"] = (
        historical_pm25
        .rolling(window)
        .std()
    )

Short-term change

In [ ]:
air["pm25_change_1d"] = air["pm25"] - air["pm25_lag_1"]
air["pm25_change_7d"] = air["pm25"] - air["pm25_lag_7"]